# Liang paired viromes: unenriched vs enriched genomic characteristics

Same analysis as `paired_virome_dataset2/bulk_vs_enriched_s15_uninducible.ipynb` genomic section,
applied to Liang PRJNA524703 paired libraries in
`liang_enriched_r6/results/` (`*_unenriched` vs `*_enriched`).

**Scope:** species-rep detections with `ictv_class == "Caudoviricetes"` and CoverM `mean ≥ 5`.

| Metric | Source |
|---|---|
| Breadth | CoverM covered bases / length |
| Capsid / tail / lysis genes per 10 kb | UHVDB metadata `num_*` / (`contig_length` / 10,000) |
| Conserved hallmarks (0–3) | MCP + TerL + portal |
| Integration / AMG–host-takeover / DNA metabolism genes per 10 kb | UHVDB protein annotations (HE6 flags) |
| Complete genomes in UHVDB species | species-cluster `complete_count` |
| Virulent score | UHVDB `virulent` |
| Proportion integrated in species | fraction of species-cluster members with `integration_status == "integrated"` |
| Host genes per 10 kb | UHVDB `host_genes` / (`contig_length` / 10,000) |
| geNomad virus score | UHVDB `virus_score` |
| geNomad hallmarks per 10 kb | UHVDB `n_hallmarks` / (`contig_length` / 10,000) |
| geNomad virus hallmark genes | UHVDB `virus_hallmarks` (hcfilter markers) |
| CheckV viral genes per 10 kb | UHVDB `viral_genes` / (`contig_length` / 10,000) |

Outputs → `liang_enriched_r6/genomic_characteristics_ge5x/`


In [ ]:
### paths + imports
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display
from scipy import stats as scipy_stats

ROOT = Path("/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB")
FIG_S15 = ROOT / "uhvdb-manuscript/figure_s15"
RESULT_DIR = (
    ROOT
    / "uhvdb-manuscript/figure_4/activity_profiling/liang_enriched_r6/results"
)
METADATA = ROOT / "uhvdb-manuscript/figure_s18/uhvdb_v5_final_metadata_v2.tsv.gz"
PROTEIN_ANNOT = ROOT / "toolkit2/databases/uhvdb/5.0/uhvdb_protein_annotations.parquet"
OUT_DIR = (
    ROOT
    / "uhvdb-manuscript/figure_4/activity_profiling/liang_enriched_r6"
    / "genomic_characteristics_ge5x"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
GENE_COUNT_CACHE = OUT_DIR / "gene_category_counts_ge80_breadth.tsv"
GENE_BREADTH_THRESHOLD = 0.8  # count genes with CoverM gene breadth ≥ this

sys.path.insert(0, str(FIG_S15))
from _he6_analysis_core import with_bio_group_flags  # noqa: E402

plt.rcParams.update({"font.size": 12})
MIN_MEAN_COV = 5.0

print("RESULT_DIR =", RESULT_DIR)
print("OUT_DIR    =", OUT_DIR)
assert RESULT_DIR.is_dir(), RESULT_DIR
assert METADATA.is_file(), METADATA
assert PROTEIN_ANNOT.is_file(), PROTEIN_ANNOT


In [ ]:
### discover samples + load UHVDB metadata
def discover_complete_samples(result_dir: Path) -> dict[str, list[str]]:
    found = {"enriched": [], "unenriched": []}
    for d in sorted(result_dir.iterdir()):
        if not d.is_dir():
            continue
        sid = d.name
        depth = d / f"{sid}.depth.tsv.gz"
        if not depth.is_file():
            continue
        if sid.endswith("_enriched"):
            found["enriched"].append(sid)
        elif sid.endswith("_unenriched"):
            found["unenriched"].append(sid)
    return found


samples = discover_complete_samples(RESULT_DIR)
print(f"complete unenriched: {len(samples['unenriched'])}")
print(f"complete enriched:   {len(samples['enriched'])}")

uhvdb_full = pl.read_csv(METADATA, separator="\t")
sp_reps = uhvdb_full.filter(pl.col("seq_name") == pl.col("seqhash_rep")).with_columns(
    pl.col("species_cluster_id").cast(pl.Int64)
)
quality = (
    uhvdb_full.with_columns(pl.col("species_cluster_id").cast(pl.Int64))
    .group_by("species_cluster_id")
    .agg(
        [
            (pl.col("checkv_quality") == "Complete").sum().alias("complete_count"),
            (pl.col("integration_status") == "integrated")
            .mean()
            .alias("prop_integrated_in_species"),
            pl.len().alias("n_members_in_species"),
        ]
    )
)
meta_sp = (
    sp_reps.select(
        [
            "uhvdb_id",
            "species_cluster_id",
            "ictv_class",
            "contig_length",
            "num_capsid",
            "num_tail",
            "num_lysis",
            "mcp_hallmark",
            "terl_hallmark",
            "portal_hallmark",
            "virulent",
            "host_genes",
            "n_genes",
            "virus_score",
            "n_hallmarks",
            "virus_hallmarks",
            "viral_genes",
        ]
    )
    .unique("uhvdb_id", keep="first")
    .join(quality, on="species_cluster_id", how="left")
    .with_columns(
        [
            pl.col("virulent").cast(pl.Float64),
            pl.col("virus_score").cast(pl.Float64),
            pl.col("n_hallmarks").cast(pl.Float64),
            pl.col("virus_hallmarks").cast(pl.Float64),
            pl.col("viral_genes").cast(pl.Float64),
            pl.col("host_genes").cast(pl.Float64),
            pl.col("prop_integrated_in_species").cast(pl.Float64),
        ]
    )
    .rename(
        {
            "mcp_hallmark": "annot_mcp_hallmark",
            "terl_hallmark": "annot_terl_hallmark",
            "portal_hallmark": "annot_portal_hallmark",
        }
    )
)
print(f"species reps in metadata: {meta_sp.height:,}")

In [ ]:
### load CoverM depth + join metadata (Caudoviricetes species-reps)
def load_depth(path: Path, sample_id: str) -> pl.DataFrame:
    raw = pl.read_csv(path, separator="\t")
    cols = raw.columns
    rename = {
        cols[0]: "contig_id",
        cols[1]: "trimmed_mean",
        cols[2]: "mean",
        cols[3]: "variance",
        cols[4]: "covered_bases",
        cols[5]: "length",
    }
    return (
        raw.rename(rename)
        .with_columns(
            [
                pl.col("trimmed_mean").cast(pl.Float64),
                pl.col("mean").cast(pl.Float64),
                pl.col("variance").cast(pl.Float64),
                pl.col("covered_bases").cast(pl.Float64),
                pl.col("length").cast(pl.Float64),
                pl.lit(sample_id).alias("sample_id"),
            ]
        )
        .with_columns((pl.col("covered_bases") / pl.col("length")).alias("breadth"))
    )


def load_library(lib: str, sample_ids: list[str]) -> pl.DataFrame:
    frames = []
    for i, sid in enumerate(sample_ids, 1):
        if i == 1 or i % 10 == 0 or i == len(sample_ids):
            print(f"  {lib}: {i}/{len(sample_ids)} {sid}", flush=True)
        frames.append(load_depth(RESULT_DIR / sid / f"{sid}.depth.tsv.gz", sid))
    coverm = pl.concat(frames, how="diagonal_relaxed")
    detected = (
        coverm.filter(pl.col("contig_id").str.starts_with("UHVDB-"))
        .filter(pl.col("breadth").fill_null(0.0) > 0)
        .join(meta_sp, left_on="contig_id", right_on="uhvdb_id", how="inner")
        .filter(pl.col("ictv_class") == "Caudoviricetes")
        .unique(["sample_id", "species_cluster_id"], keep="first")
    )
    print(
        f"  {lib}: {detected.height:,} Caudoviricetes species-rep detections "
        f"across {detected['sample_id'].n_unique()} samples"
    )
    return detected


print("Loading unenriched...")
une_det = load_library("unenriched", samples["unenriched"])
print("Loading enriched...")
enr_det = load_library("enriched", samples["enriched"])

In [ ]:
### gene-category counts from gene_coverage (≥80% breadth) + ≥5× feature table
# Capsid/tail/lysis/integration/AMG/DNA metabolism and MCP+TerL+portal are counted
# from sample *.gene_coverage.tsv.gz genes with breadth ≥ GENE_BREADTH_THRESHOLD.
# Static genome annotations are NOT used for those gene-category metrics.


def gene_coverage_category_counts(detections: pl.DataFrame) -> pl.DataFrame:
    """Per (sample_id, contig_id) gene-category counts among genes with breadth ≥ threshold."""
    if GENE_COUNT_CACHE.is_file():
        cached = pl.read_csv(GENE_COUNT_CACHE, separator="\t")
        need = detections.select(["sample_id", "contig_id"]).unique()
        hit = need.join(cached, on=["sample_id", "contig_id"], how="left")
        if hit.filter(pl.col("num_lysis").is_null()).height == 0:
            print(f"gene counts: cache hit ({cached.height:,} detection rows)")
            return cached
        print("gene counts: cache incomplete; recomputing")

    pairs_by_sample = {}
    for g in detections.select(["sample_id", "contig_id"]).unique().partition_by(
        "sample_id", as_dict=False
    ):
        pairs_by_sample[g["sample_id"][0]] = set(g["contig_id"].to_list())

    parts = []
    for sid, contig_ids in sorted(pairs_by_sample.items()):
        path = RESULT_DIR / sid / f"{sid}.gene_coverage.tsv.gz"
        if not path.is_file():
            raise FileNotFoundError(path)
        base_pairs = pl.DataFrame(
            {"sample_id": [sid] * len(contig_ids), "contig_id": list(contig_ids)}
        )
        gc = (
            pl.read_csv(
                path,
                separator="\t",
                columns=[
                    "genomovar_rep",
                    "pharokka_annot",
                    "pharokka_category",
                    "phold_category",
                    "empathi_annot",
                    "breadth",
                ],
            )
            .filter(
                (pl.col("breadth") >= GENE_BREADTH_THRESHOLD)
                & pl.col("genomovar_rep").is_in(list(contig_ids))
            )
            .with_columns(pl.lit(sid).alias("sample_id"))
        )
        if gc.height == 0:
            parts.append(
                base_pairs.with_columns(
                    [
                        pl.lit(0).alias(c)
                        for c in [
                            "n_genes_breadth_ge80",
                            "num_capsid",
                            "num_tail",
                            "num_lysis",
                            "n_integration",
                            "n_amg_host_takeover",
                            "n_dna_metabolism",
                            "annot_mcp_hallmark",
                            "annot_terl_hallmark",
                            "annot_portal_hallmark",
                            "n_conserved_hallmarks",
                        ]
                    ]
                )
            )
            continue
        flagged = with_bio_group_flags(gc)
        agg = (
            flagged.group_by(["sample_id", "genomovar_rep"])
            .agg(
                [
                    pl.len().alias("n_genes_breadth_ge80"),
                    pl.col("is_capsid_packaging").sum().cast(pl.Float64).alias("num_capsid"),
                    pl.col("is_tail").sum().cast(pl.Float64).alias("num_tail"),
                    pl.col("is_lysis").sum().cast(pl.Float64).alias("num_lysis"),
                    pl.col("is_integration").sum().cast(pl.Float64).alias("n_integration"),
                    pl.col("is_amg_host_takeover")
                    .sum()
                    .cast(pl.Float64)
                    .alias("n_amg_host_takeover"),
                    pl.col("is_dna_metabolism")
                    .sum()
                    .cast(pl.Float64)
                    .alias("n_dna_metabolism"),
                    (
                        ((pl.col("pharokka_annot") == "major head protein").sum() >= 1)
                        | ((pl.col("phold_category") == "major head protein").sum() >= 1)
                        | ((pl.col("empathi_annot") == "pvp|capsid|major_capsid").sum() >= 1)
                    )
                    .cast(pl.Int64)
                    .alias("annot_mcp_hallmark"),
                    (
                        ((pl.col("pharokka_annot") == "terminase large subunit").sum() >= 1)
                        | ((pl.col("phold_category") == "terminase large subunit").sum() >= 1)
                        | (
                            (
                                pl.col("empathi_annot")
                                == "DNA-associated|terminase|packaging_assembly"
                            ).sum()
                            >= 1
                        )
                    )
                    .cast(pl.Int64)
                    .alias("annot_terl_hallmark"),
                    (
                        ((pl.col("pharokka_annot") == "portal protein").sum() >= 1)
                        | ((pl.col("phold_category") == "portal protein").sum() >= 1)
                        | ((pl.col("empathi_annot") == "pvp|portal").sum() >= 1)
                    )
                    .cast(pl.Int64)
                    .alias("annot_portal_hallmark"),
                ]
            )
            .rename({"genomovar_rep": "contig_id"})
            .with_columns(
                (
                    pl.col("annot_mcp_hallmark")
                    + pl.col("annot_terl_hallmark")
                    + pl.col("annot_portal_hallmark")
                ).alias("n_conserved_hallmarks")
            )
        )
        parts.append(
            base_pairs.join(agg, on=["sample_id", "contig_id"], how="left").with_columns(
                [
                    pl.col(c).fill_null(0)
                    for c in [
                        "n_genes_breadth_ge80",
                        "num_capsid",
                        "num_tail",
                        "num_lysis",
                        "n_integration",
                        "n_amg_host_takeover",
                        "n_dna_metabolism",
                        "annot_mcp_hallmark",
                        "annot_terl_hallmark",
                        "annot_portal_hallmark",
                        "n_conserved_hallmarks",
                    ]
                ]
            )
        )

    out = pl.concat(parts, how="vertical_relaxed")
    out.write_csv(GENE_COUNT_CACHE, separator="\t")
    print(f"gene counts: wrote {GENE_COUNT_CACHE} ({out.height:,})")
    return out


hi = pl.concat(
    [
        une_det.filter(pl.col("mean").fill_null(0.0) >= MIN_MEAN_COV),
        enr_det.filter(pl.col("mean").fill_null(0.0) >= MIN_MEAN_COV),
    ],
    how="diagonal_relaxed",
)
gene_counts = gene_coverage_category_counts(hi)


def prepare_genomic(df: pl.DataFrame, lib_type: str) -> pl.DataFrame:
    # Drop annotation-based gene counts before joining coverage-based counts
    drop_annot = [
        c
        for c in (
            "num_capsid",
            "num_tail",
            "num_lysis",
            "n_integration",
            "n_amg_host_takeover",
            "n_dna_metabolism",
            "annot_mcp_hallmark",
            "annot_terl_hallmark",
            "annot_portal_hallmark",
        )
        if c in df.columns
    ]
    base = df.filter(pl.col("mean").fill_null(0.0) >= MIN_MEAN_COV)
    if drop_annot:
        base = base.drop(drop_annot)
    return (
        base.join(gene_counts, on=["sample_id", "contig_id"], how="left")
        .with_columns(
            [
                pl.lit(lib_type).alias("library"),
                pl.col("contig_length").cast(pl.Float64).alias("genome_length"),
                pl.col("num_capsid").fill_null(0).cast(pl.Float64),
                pl.col("num_tail").fill_null(0).cast(pl.Float64),
                pl.col("num_lysis").fill_null(0).cast(pl.Float64),
                pl.col("complete_count").cast(pl.Float64),
                pl.col("virulent").cast(pl.Float64),
                pl.col("prop_integrated_in_species").cast(pl.Float64),
                pl.col("host_genes").cast(pl.Float64),
                pl.col("virus_score").cast(pl.Float64),
                pl.col("n_hallmarks").cast(pl.Float64),
                pl.col("virus_hallmarks").cast(pl.Float64),
                pl.col("viral_genes").cast(pl.Float64),
                pl.col("n_integration").fill_null(0).cast(pl.Float64),
                pl.col("n_amg_host_takeover").fill_null(0).cast(pl.Float64),
                pl.col("n_dna_metabolism").fill_null(0).cast(pl.Float64),
                pl.col("n_conserved_hallmarks").fill_null(0).cast(pl.Float64),
            ]
        )
        .with_columns(
            [
                (pl.col("num_capsid") / (pl.col("genome_length") / 10_000)).alias(
                    "capsid_per_10kb"
                ),
                (pl.col("num_tail") / (pl.col("genome_length") / 10_000)).alias(
                    "tail_per_10kb"
                ),
                (pl.col("num_lysis") / (pl.col("genome_length") / 10_000)).alias(
                    "lysis_per_10kb"
                ),
                (pl.col("n_integration") / (pl.col("genome_length") / 10_000)).alias(
                    "integration_per_10kb"
                ),
                (
                    pl.col("n_amg_host_takeover")
                    / (pl.col("genome_length") / 10_000)
                ).alias("amg_host_takeover_per_10kb"),
                (
                    pl.col("n_dna_metabolism") / (pl.col("genome_length") / 10_000)
                ).alias("dna_metabolism_per_10kb"),
                (pl.col("n_hallmarks") / (pl.col("genome_length") / 10_000)).alias(
                    "n_hallmarks_per_10kb"
                ),
                (pl.col("viral_genes") / (pl.col("genome_length") / 10_000)).alias(
                    "viral_genes_per_10kb"
                ),
                (pl.col("host_genes") / (pl.col("genome_length") / 10_000)).alias(
                    "host_genes_per_10kb"
                ),
            ]
        )
    )


genomic = pl.concat(
    [
        prepare_genomic(une_det, "unenriched"),
        prepare_genomic(enr_det, "enriched"),
    ],
    how="diagonal_relaxed",
)
print(
    genomic.group_by("library")
    .agg(
        [
            pl.len().alias("n_detections"),
            pl.col("contig_id").n_unique().alias("n_species"),
            pl.col("sample_id").n_unique().alias("n_samples"),
            pl.col("mean").median().alias("median_mean_cov"),
            pl.col("breadth").median().alias("median_breadth"),
        ]
    )
    .sort("library")
)
genomic.write_csv(OUT_DIR / "detections_ge5x.tsv", separator="\t")
print("wrote", OUT_DIR / "detections_ge5x.tsv")
print(
    f"Gene categories use CoverM gene_coverage breadth ≥ {GENE_BREADTH_THRESHOLD}"
)


In [ ]:
### summary stats + Mann–Whitney U (unenriched vs enriched)
METRIC_SPECS = [
    ("breadth", "Breadth"),
    ("capsid_per_10kb", "Capsid genes / 10 kb"),
    ("tail_per_10kb", "Tail genes / 10 kb"),
    ("lysis_per_10kb", "Lysis genes / 10 kb"),
    ("n_conserved_hallmarks", "Conserved hallmarks (MCP+TerL+portal)"),
    ("integration_per_10kb", "Integration genes / 10 kb"),
    ("amg_host_takeover_per_10kb", "AMG / host-takeover genes / 10 kb"),
    ("dna_metabolism_per_10kb", "DNA metabolism genes / 10 kb"),
    ("complete_count", "Complete genomes in UHVDB species"),
    ("virulent", "Virulent score"),
    ("prop_integrated_in_species", "Proportion integrated in species"),
    ("host_genes_per_10kb", "Host genes / 10 kb"),
    ("virus_score", "geNomad virus score"),
    ("n_hallmarks_per_10kb", "geNomad hallmarks / 10 kb"),
    ("virus_hallmarks", "geNomad virus hallmark genes"),
    ("viral_genes_per_10kb", "CheckV viral genes / 10 kb"),
]

LIB_ORDER = ("unenriched", "enriched")
LIB_LABELS = {"unenriched": "Unenriched", "enriched": "Enriched"}


def _vals(df: pl.DataFrame, col: str) -> np.ndarray:
    return (
        df.select(pl.col(col).cast(pl.Float64))
        .drop_nulls()
        .to_series()
        .to_numpy()
    )


def _bh_fdr(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, dtype=float)
    q = np.full(p.shape, np.nan)
    ok = np.isfinite(p)
    if not ok.any():
        return q
    pv = p[ok]
    m = len(pv)
    order = np.argsort(pv)
    ranked = pv[order]
    q_sorted = ranked * m / np.arange(1, m + 1)
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
    q_sorted = np.clip(q_sorted, 0, 1)
    q[ok] = q_sorted[np.argsort(order)]
    return q


une_g = genomic.filter(pl.col("library") == "unenriched")
enr_g = genomic.filter(pl.col("library") == "enriched")

rows = []
for col, label in METRIC_SPECS:
    a = _vals(une_g, col)
    b = _vals(enr_g, col)
    if len(a) == 0 or len(b) == 0:
        stat, p, cliffs_d = np.nan, np.nan, np.nan
    else:
        stat, p = scipy_stats.mannwhitneyu(a, b, alternative="two-sided")
        # positive δ = enriched > unenriched
        cliffs_d = -((2.0 * stat) / (len(a) * len(b)) - 1.0)
    rows.append(
        {
            "metric": col,
            "label": label,
            "n_unenriched": int(len(a)),
            "n_enriched": int(len(b)),
            "median_unenriched": float(np.median(a)) if len(a) else np.nan,
            "median_enriched": float(np.median(b)) if len(b) else np.nan,
            "mean_unenriched": float(np.mean(a)) if len(a) else np.nan,
            "mean_enriched": float(np.mean(b)) if len(b) else np.nan,
            "mannwhitney_U": float(stat),
            "p_value": float(p),
            "cliffs_delta_enriched_minus_unenriched": float(cliffs_d),
        }
    )

stats_df = pd.DataFrame(rows)
stats_df["q_value_bh"] = _bh_fdr(stats_df["p_value"].to_numpy())
stats_df["significant_q0.05"] = stats_df["q_value_bh"] < 0.05
display(stats_df.round(4))
stats_df.to_csv(OUT_DIR / "unenriched_vs_enriched_stats.tsv", sep="\t", index=False)
print("wrote", OUT_DIR / "unenriched_vs_enriched_stats.tsv")

In [ ]:
### boxplots: unenriched vs enriched genomic characteristics
pal = {"unenriched": "#4C72B0", "enriched": "#DD8452"}
# Sort panels by ascending p-value
_label_map = dict(METRIC_SPECS)
plot_metrics = [
    (m, _label_map[m])
    for m in stats_df.sort_values("p_value")["metric"].tolist()
    if m in _label_map
]
n_metrics = len(plot_metrics)
ncol = 3
nrow = int(np.ceil(n_metrics / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.6 * nrow), squeeze=False)
qmap = dict(zip(stats_df["metric"], stats_df["q_value_bh"]))
dmap = dict(
    zip(stats_df["metric"], stats_df["cliffs_delta_enriched_minus_unenriched"])
)

for ax, (col, label) in zip(axes.ravel(), plot_metrics):
    data = [_vals(une_g, col), _vals(enr_g, col)]
    compress_axis = col == "complete_count"
    bp = ax.boxplot(
        data,
        positions=[0, 1],
        widths=0.55,
        patch_artist=True,
        showfliers=False,
        showmeans=not compress_axis,
        meanprops=dict(
            marker="D",
            markersize=6,
            markerfacecolor="white",
            markeredgecolor="black",
            markeredgewidth=1.0,
        ),
        flierprops=dict(
            marker="o",
            markersize=3,
            markerfacecolor="0.5",
            markeredgecolor="none",
            alpha=0.35,
        ),
        medianprops=dict(color="black", linewidth=1.4),
        whiskerprops=dict(color="black", linewidth=1.0),
        capprops=dict(color="black", linewidth=1.0),
        boxprops=dict(linewidth=1.0, edgecolor="black"),
    )
    for patch, lib in zip(bp["boxes"], LIB_ORDER):
        patch.set_facecolor(pal[lib])
        patch.set_alpha(0.85)
    if compress_axis:
        box_tops = [np.nanmax(box.get_path().vertices[:, 1]) for box in bp["boxes"]]
        box_bots = [np.nanmin(box.get_path().vertices[:, 1]) for box in bp["boxes"]]
        y_min, y_max = float(min(box_bots)), float(max(box_tops))
    else:
        whisk_ys = np.concatenate([w.get_ydata() for w in bp["whiskers"]])
        y_min, y_max = float(np.nanmin(whisk_ys)), float(np.nanmax(whisk_ys))
    if y_max <= y_min:
        y_max = y_min + 1.0
    pad = 0.06 * (y_max - y_min + 1e-9)
    for xpos, arr in zip((0, 1), data):
        mu = float(np.mean(arr))
        ax.text(
            xpos,
            y_max + pad,
            f"μ={mu:.2g}",
            ha="center",
            va="bottom",
            fontsize=9,
            color="0.2",
        )
    ax.set_ylim(y_min - pad, y_max + 4.5 * pad)
    q = qmap.get(col, np.nan)
    cliff_d = dmap.get(col, np.nan)
    sig = (
        "***"
        if q < 1e-3
        else "**"
        if q < 1e-2
        else "*"
        if q < 5e-2
        else "ns"
    )
    ax.set_xticks([0, 1], [LIB_LABELS[x] for x in LIB_ORDER])
    ax.set_ylabel(label)
    ax.set_title(f"q={q:.2g}  δ={cliff_d:+.2f}  {sig}", fontsize=11)
    ax.grid(axis="y", alpha=0.3)

for ax in axes.ravel()[n_metrics:]:
    ax.axis("off")

n_une, n_enr = une_g.height, enr_g.height
fig.suptitle(
    f"Liang Caudoviricetes detections with mean coverage ≥ {MIN_MEAN_COV:.0f}×\n"
    f"n = {n_une:,} unenriched  ·  {n_enr:,} enriched",
    y=1.02,
    fontsize=14,
)
fig.tight_layout()
png_path = OUT_DIR / "unenriched_vs_enriched_genomic_characteristics.png"
pdf_path = OUT_DIR / "unenriched_vs_enriched_genomic_characteristics.pdf"
fig.savefig(png_path, dpi=200, bbox_inches="tight")
try:
    fig.savefig(pdf_path, bbox_inches="tight")
    print("wrote", pdf_path)
except Exception as exc:
    print(f"PDF save skipped ({exc.__class__.__name__}: {exc})")
plt.show()
print("wrote", png_path)